In [1]:
import csv
import re
import time
from pathlib import Path

In [2]:
file_path = Path.cwd()/"IMDB Dataset.csv"

In [3]:
def open_file(file:Path):
    """
    This function open the dataset file using csv module.
    It returns list[(review, sentiment)] or raise FileNotFound error
    """
    reviews = []
    try:
        with open(file, mode="r", newline="", encoding="utf-8") as data_file:
            data = csv.DictReader(data_file)
            for row in data:
                text = row["review"]
                label = row["sentiment"]
                reviews.append((text, label))
    except FileNotFoundError as error:
        raise FileNotFoundError(f"File not found! Check file_path: {file}") from error
    return reviews

In [4]:
data = open_file(file=file_path)
data[:2]

[("One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the 

In [5]:
def number_reviews(data:list):
    """
    This function returns the number of positive and negative reviews
    return nbr_positive_reviews, nbr_negative_reviews
    """
    nbr_positive_reviews = len([review[0] for review in data if review[1] == "positive"])
    nbr_negative_reviews = len([review[0] for review in data if review[1] == "negative"])
    return nbr_positive_reviews, nbr_negative_reviews
number_reviews(data=data)

(25000, 25000)

### In this implementation, we chose to find the conditional positive probability of all the two lists chosen.

### We start with the positive_words list then We complete with the negative_words list

In [6]:
# choosing the keyword
positive_words = ["wonderful","masterpiece", "brilliant", "love"]
negative_words = ["boring", "dreadful", "disgraceful", "painful"]

In [7]:
def contain_keyword(text:str, keyword:str) -> bool:
    """
    This is an helper function to handle reviews
    Args:
        text:str (The part of the review to filter to find the word)
        keyword:str (The word to look for)
    return True if found overwise False
    """
    return bool(re.search(rf"\b{re.escape(keyword)}\b", text, re.IGNORECASE))


In [8]:
def count_reviews_with_keyword(data, keyword:str, sentiment_type:str) -> int:
     """
    Helper function to find the number of times the keyword appears in reviews
    of the given sentiment_type (works for either "positive" or "negative").
    Args:
        data: the IMDB dataset for this problem
        keyword:str (the word to look for)
        sentiment_type: Either positive or negative
    """
     sentiment_type = sentiment_type.lower()
     count = 0
     for review in data:
          if review[1] == sentiment_type and contain_keyword(text=review[0], keyword=keyword):
               count += 1
     return count

In [9]:
# implementation example
count_reviews_with_keyword(data=data,
                           keyword=positive_words[0],
                           sentiment_type="positive"
                           )

2258

In [10]:
def probability_keyword_given_sentiment(keyword_count:int, sentiment_type:str) -> float:
    """
    This helper function finds the likelihood P(keyword | sentiment_type) of 
    finding a keyword in a review of the given sentiment_type 
    (works for either "positive" or "negative")
    Args:
        keyword_count: int = The number of time the word is found in a particular review (positive or negative reviews).
        This value is retrieved from the count_reviews_with_keyword function.
        sentiment_type:str = type of review (either positive or negative)
    return probability

    """
    sentiment_type = sentiment_type.lower()
    sentiment_index = 0 if sentiment_type == "positive" else 1
    probability = (keyword_count/number_reviews(data=data)[sentiment_index])
    return probability

In [11]:
probability_keyword_given_sentiment(
    keyword_count=count_reviews_with_keyword(data=data, keyword=positive_words[0],sentiment_type="positive"),
    sentiment_type="positive"
    )

0.09032

In [12]:
def probability_complement(probability:float) -> float:
    """
    Helper function to get the complement of a probability, P(not event) = 1 - P(event)
    (works for either a positive- or negative-conditioned probability)
    Args:
        probability:float = the probability obtained by running probability_keyword_given_sentiment function
    return complement probability
    """
    return 1 - probability

In [13]:
# Implementation example for complement probability
probability_complement(
    probability=probability_keyword_given_sentiment(
        keyword_count=count_reviews_with_keyword(data=data, keyword=positive_words[0], sentiment_type="positive"),
        sentiment_type="positive"
    ))

0.90968

In [14]:
def probability_keyword_marginal(data, keyword:str) -> float:
    """
    Helper function to get P(keyword): the marginal probability
    of the keyword appearing anywhere in the dataset, regardless of sentiment
    Args:
        data: the dataset provided
        keyword:str = the keyword to look for
    return marginal_probability
    """
    count = 0
    for review in data:
        if contain_keyword(text=review[0], keyword=keyword):
            count +=1
    probability = count/len(data)
    return probability

In [15]:
# implementation example
probability_keyword_marginal(data=data, keyword=positive_words[0])

0.0556

In [16]:
positive_review = (number_reviews(data=data)[0]/len(data))*100
negative_review = (number_reviews(data=data)[1]/len(data))*100
print(f"probability of finding a positive review in the dataset is: {positive_review:.2f}%")
print(f"probability of finding a negative review in the dataset is: {negative_review:.2f}%")

probability of finding a positive review in the dataset is: 50.00%
probability of finding a negative review in the dataset is: 50.00%


In [17]:
def bayesian_probability(data, keyword: str, sentiment_type: str) -> float:
    """
    Computes P(sentiment_type | keyword) via Bayes' theorem. This function reuses
    all helpers functions, work for either 'positive' or 'negative'
    Args:
        data = The provided dataset
        keyword:str = The word to look for
        sentiment_type = The review type (positive or negative)
    return conditional_probability
    """
    sentiment_index = 0 if sentiment_type == "positive" else 1

    keyword_count = count_reviews_with_keyword(data=data, keyword=keyword, sentiment_type=sentiment_type)

    p_keyword_given_sentiment = probability_keyword_given_sentiment(keyword_count=keyword_count, sentiment_type=sentiment_type)
    p_sentiment = number_reviews(data=data)[sentiment_index] / len(data)
    p_keyword = probability_keyword_marginal(data=data, keyword=keyword)

    try:
        conditional_probability = (p_keyword_given_sentiment * p_sentiment) / p_keyword
    except ZeroDivisionError:
        conditional_probability = 0

    return {
        "keyword": keyword,
        "prior_probability in %":p_sentiment,
        "likelihood in %":p_keyword_given_sentiment,
        "marginal_probability in %":p_keyword,
        "posterior in %":conditional_probability
    }


In [18]:
print("Display all posterior probabilities ")
start = time.time()
probability_table_df = []
for keyword in positive_words:
    probability = bayesian_probability(data=data, keyword=keyword, sentiment_type="positive")
    probability_table_df.append(probability)
    print(f"P(positive | '{keyword}') = {probability["posterior in %"]:.3f} ({probability["posterior in %"]*100:.2f}%)")
end = time.time()
print(f"\nCSV-based computation took {end - start:.3f} seconds")

Display all posterior probabilities 
P(positive | 'wonderful') = 0.812 (81.22%)
P(positive | 'masterpiece') = 0.727 (72.74%)
P(positive | 'brilliant') = 0.760 (76.01%)
P(positive | 'love') = 0.635 (63.49%)

CSV-based computation took 8.474 seconds


In [19]:
# run table of values
import pandas as pd

percentage_table_df = pd.DataFrame(data=probability_table_df)
probability_table = percentage_table_df.set_index("keyword") * 100
probability_table.round(2)



,prior_probability in %,likelihood in %,marginal_probability in %,posterior in %
keyword,,,,
wonderful,50.0,9.03,5.56,81.22
masterpiece,50.0,3.51,2.41,72.74
brilliant,50.0,6.35,4.18,76.01
love,50.0,22.66,17.85,63.49
